# TOFU Llama 3.1 8B NPO-LoRA evaluation recovery

This notebook only evaluates an already-trained adapter from the completed Kaggle v2 artifact. It will refuse to run unless the archive proves the exact preregistered protocol, 500 optimizer steps, seed 42, valid adapter hashes, and verified 8B baseline inputs. It does **not** train or resume training.

Before running: attach the completed kernel output ZIP as a private Kaggle dataset, enable Internet, and select GPU T4 x2. If the original run already contains complete adapter evaluation files, this notebook skips inference and independently verifies them. Otherwise it runs only the missing evaluation, streams progress to the notebook output, then runs the independent verifier.

In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = Path("/kaggle/working/requirements-npo-pilot.txt")
requirements.write_text("transformers==5.22.1\naccelerate==1.15.0\npeft==0.21.2\nscipy==1.16.3\nrouge-score==0.1.2\n", encoding="utf-8")
expected = {"transformers": "5.22.1", "accelerate": "1.15.0", "peft": "0.21.2", "scipy": "1.16.3", "rouge-score": "0.1.2"}
def versions_now():
    found = {}
    for package in expected:
        try:
            found[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            found[package] = None
    return found
if versions_now() != expected:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(requirements)], check=True)
if versions_now() != expected:
    raise RuntimeError(f"Pinned package check failed: {versions_now()}")
try:
    metadata.version("torchao")
except metadata.PackageNotFoundError:
    pass
else:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "--yes", "torchao"], check=True, capture_output=True, text=True)
    try:
        metadata.version("torchao")
    except metadata.PackageNotFoundError:
        pass
    else:
        raise RuntimeError("torchao remained installed after removal")
if importlib.util.find_spec("torchao") is not None:
    raise RuntimeError("torchao module remains importable after removal")
import torch
from transformers import LlamaConfig, LlamaForCausalLM
from peft import LoraConfig, get_peft_model
tiny = LlamaForCausalLM(LlamaConfig(vocab_size=32, hidden_size=16, intermediate_size=32, num_hidden_layers=1, num_attention_heads=2, num_key_value_heads=2))
tiny = get_peft_model(tiny, LoraConfig(r=2, lora_alpha=4, lora_dropout=0.0, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], bias="none", task_type="CAUSAL_LM"))
tiny_ids = torch.tensor([[1, 2, 3, 4]])
tiny_loss = tiny(input_ids=tiny_ids, labels=tiny_ids, use_cache=False).loss
tiny_loss.backward()
tiny_trainable = [parameter for parameter in tiny.parameters() if parameter.requires_grad]
tiny_grad = sum(float(parameter.grad.float().abs().sum()) for parameter in tiny_trainable if parameter.grad is not None)
if not torch.isfinite(tiny_loss) or not tiny_trainable or not torch.isfinite(torch.tensor(tiny_grad)) or tiny_grad <= 0:
    raise RuntimeError("PEFT LoRA dispatch preflight failed")
print({"packages": versions_now(), "torch": metadata.version("torch"), "torchao": "absent", "peft_lora_dispatch_smoke": "PASS", "tiny_loss": float(tiny_loss.detach()), "tiny_adapter_gradient_l1": tiny_grad}, flush=True)

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import torch

PROTOCOL = json.loads('{"adaptation": {"determinism_note": "Python, CPU Torch, and CUDA Torch RNGs are seeded; CUDA kernels may remain nondeterministic.", "effective_paired_batch_size": 8, "epochs": 10, "forget_retain_pairing": "one uniformly sampled retain example with replacement for each forget example; seed 42", "gradient_clip_norm": 1.0, "hardware": "Kaggle GPU T4 x2 only; refuse CPU/disk weight offload; FP16 base, FP32 adapter and loss; SDPA", "learning_rate": 1e-05, "lora_alpha": 32, "lora_dropout": 0.0, "lora_rank": 16, "max_length": 512, "optimizer": "AdamW, betas=(0.9,0.999), eps=1e-8, weight_decay=0", "precision": "FP16 frozen base checkpoint; FP32 LoRA parameters and loss; SDPA", "scheduler": "linear; one epoch warmup", "scope": "LoRA adapters only; frozen FP16 base; report as an NPO-LoRA adaptation, not full-parameter NPO.", "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], "truncation": false, "warmup": "one epoch of optimizer steps, then linear decay"}, "base_checkpoint": {"architecture_id": "meta-llama/Llama-3.1-8B-Instruct", "repo_id": "open-unlearning/tofu_Llama-3.1-8B-Instruct_full", "revision": "1a5c5b1a557f8c99bdadecd5168ebd03f640b00e"}, "bounded_smoke": {"optimizer_steps": 1, "real_data_rows": "first eight pinned forget10 rows and one sampled retain row", "required_before_full_pilot": true, "success_criteria": ["finite NPO/retain losses and gradients", "exactly one optimizer step", "nonzero LoRA parameter update", "T4 x2 placement", "no CPU/disk offload", "smoke adapter discarded and never scored as research result"]}, "code_sources": {"adapter_evaluator": "evaluate_npo_adapter.py", "independent_run_verifier": "verify_npo_run.py", "objective": "npo_objective.py", "objective_verifier": "verify_npo_objective.py", "official_evaluator": "tofu_official_eval.py", "trainer": "npo_lora_train.py"}, "dataset": {"evaluation_file_sha256": {"forget10.json": "0044c8c2e70a38be93f62ec6cb1c1cc2a1f55a8df2fb549a4da2da8dde9d92f6", "forget10_perturbed.json": "6fbcb946c57ea1d7b2124cea0e61bf3b5409d1bc10d02368f090109450ed73c7", "holdout10.json": "efec32c7a7f66bfefaa90b8a4cc583296b0bbf882495db61076dec2d6edf44dd", "real_authors_perturbed.json": "a23255383ac75a5fbe5acd57622a03fb9c368954fd7b2428afa152b784d9533a", "retain90.json": "debcf019c41db7b30b62ca6ad41fd8a4bcd29a5b321c21d2961974e308dfca55", "retain_perturbed.json": "fc69f33bad70d3dca65920bdb54380039e3e8872dd16bbae20c1a839b99533fb", "world_facts_perturbed.json": "0e171838040d0ec94b1ce248d891508e85e3b77344370077caa27b61898eb3e1"}, "evaluation_rows": {"forget10": 400, "holdout10": 400, "real_authors": 100, "retain90": 400, "world_facts": 117}, "repo_id": "locuslab/TOFU", "revision": "324592d84ae4f482ac7249b9285c2ecdb53e3a68", "training": {"forget_file": {"name": "forget10.json", "rows": 400, "sha256": "0044c8c2e70a38be93f62ec6cb1c1cc2a1f55a8df2fb549a4da2da8dde9d92f6"}, "holdout_used_for_training": false, "retain_file": {"name": "retain90.json", "rows": 3600, "sha256": "debcf019c41db7b30b62ca6ad41fd8a4bcd29a5b321c21d2961974e308dfca55"}}}, "evaluation": {"official_metrics": ["answer_probability", "greedy-generation ROUGE-L recall", "Truth Ratio score", "normalized option probability", "nine-component model-utility harmonic mean"], "primary_direction": "positive values indicate movement toward the matching retain90 oracle; interpret together with the measured full-to-oracle contrast and utility gates.", "primary_metric": "forget10 mean Truth Ratio score, paired NPO adapter minus the same Llama 3.1 8B full-checkpoint rows", "primary_signal_rule": "Exploratory signal only if the 95% profile-block bootstrap interval for NPO-minus-full Truth Ratio score is wholly above zero.", "secondary_descriptive_metrics": ["forget10 answer probability and ROUGE-L change", "NPO-versus-oracle paired contrasts", "all individual utility components", "profile-cluster intervals"], "uncertainty": "paired percentile bootstrap resampling the 20 TOFU profile blocks, 5000 draws, seed 42; preserve all rows in each sampled block", "utility_gates": {"holdout10.answer_probability": {"minimum_delta": -0.05, "rule": "95% paired profile-block interval lower bound strictly greater than -0.05"}, "model_utility.harmonic_mean": {"minimum_delta": -0.05, "rule": "point difference strictly greater than -0.05; descriptive gate because utility rows do not share profile IDs"}, "retain90.answer_probability": {"minimum_delta": -0.05, "rule": "95% paired profile-block interval lower bound strictly greater than -0.05"}, "retain90.rougeL_recall": {"minimum_delta": -0.05, "rule": "95% paired profile-block interval lower bound strictly greater than -0.05"}}}, "interpretation_limits": ["Single seed and one model family: exploratory only, cannot establish general effectiveness.", "Adapter training changes capacity and optimization relative to the original full-parameter recipe.", "TOFU profiles are fictional and do not support claims about deletion of real personal data or real-world privacy.", "Truth Ratio is one benchmark metric; this run is not proof of complete knowledge removal.", "The 8B and 3B checkpoints have different architectures and training histories; comparing their single-seed results does not isolate parameter count as a cause.", "This one-seed pilot is an implementation and feasibility diagnostic, not confirmatory evidence."], "model_family": "8b", "objective": {"beta": 0.1, "equivalence_requirement": "Pinned upstream loss value and parameter gradients match the candidate implementation within absolute tolerance 1e-6 on the deterministic CPU fixture before any GPU training.", "forget_weight_gamma": 1.0, "formula": "mean[-2/beta * log_sigmoid(beta * (current_sequence_summed_nll - reference_sequence_summed_nll))] + retain_token_mean_nll", "method": "NPO forget objective plus retain negative log-likelihood", "reference_model": "frozen initial full checkpoint; reference sequence NLL precomputed before LoRA insertion", "retain_weight_alpha": 1.0, "source": "locuslab/open-unlearning@2d98027677f3468d52b6baa3fc4f2b9f2dc08820"}, "protocol_id": "tofu_8b_npo_lora_pilot_v1", "registered_locally_at": "2026-11-08", "research_question": "On TOFU forget10, does the fixed, resource-bounded LoRA implementation of the pinned Open-Unlearning NPO objective move the Llama 3.1 8B checkpoint toward its matching retain90 oracle while preserving retain, holdout, real-author, and world-fact utility?", "retain_only_reference_checkpoint": {"repo_id": "open-unlearning/tofu_Llama-3.1-8B-Instruct_retain90", "revision": "f02956a9f25a18f38ad7808fbefb2f957347273e"}, "seed": 42, "status": "PREREGISTERED_BEFORE_GPU_RUN", "stopping_rules": ["Do not train unless the pinned upstream objective/gradient check passes and the exact 8B full/oracle baseline outputs independently verify.", "Do not train unless the bounded real-data 8B smoke passes with a nonzero adapter update and GPU-only placement on two T4 GPUs.", "Stop and preserve partial receipts on non-finite loss/gradient, OOM, missing data, revision/hash mismatch, truncation, offload, or incomplete optimizer steps.", "Do not tune after inspecting pilot evaluation results; any change requires a new protocol and new seed.", "A failed primary or utility gate is a diagnostic result, not a reason to change the threshold or discard the run."], "study_type": "Exploratory one-seed 8B NPO-LoRA implementation pilot; not confirmatory evidence and not a controlled scaling comparison."}')
OBJECTIVE_CHECK = json.loads('{"absolute_loss_difference": 0.0, "candidate_loss": 13.862943649291992, "candidate_objective_sha256": "f597000adc9fc325d1ef9fc23debcd51f6ead13c266738718884631738248af9", "combined_vs_sequential_backward_max_gradient_difference": 0.0, "examples": 2, "loss_reduction": "sum token NLL per sequence; batch mean for NPO; token mean for retain NLL", "maximum_parameter_gradient_difference": 6.705522537231445e-08, "npo_beta": 0.1, "protocol_sha256": "17de753d3c92539213eb74962ef124677cf0ed8f4f92e9f390c8e62197ddcb1b", "python_version": "3.14.6 (main, Jun 10 2026, 10:03:53) [Clang 17.0.0 (clang-1700.6.4.2)]", "retain_nll_loss_difference": 0.0, "status": "PASS", "torch_version": "2.13.0", "upstream": "locuslab/open-unlearning@2d98027677f3468d52b6baa3fc4f2b9f2dc08820", "upstream_loss": 13.862943649291992, "upstream_utils_sha256": "742bd2682b5829bb731988c952550c9c2ff75889737f8eee14f1b148d6d975e9"}')
EXPECTED_PROTOCOL_SHA256 = '17de753d3c92539213eb74962ef124677cf0ed8f4f92e9f390c8e62197ddcb1b'
EXPECTED_BASELINE_MANIFEST_SHA256 = '4e017b430e55702715d0ebe1cb2c573197a29c18b8e928cc90601291d71e9e2f'
EXPECTED_SOURCE_SHA256 = json.loads('{"evaluate_npo_adapter.py": "bca3638bb65f64486c9470e5f0320b7424e4eae79758c5787023d7472df667b0", "npo_lora_train.py": "9e7be7beb1316cb4e0088951c003a937f38de713812d31c51816b7543486b37e", "npo_objective.py": "f597000adc9fc325d1ef9fc23debcd51f6ead13c266738718884631738248af9", "tofu_official_eval.py": "8a3fc9e38a810aba45c46f3940948d913adaf11e60c750ee0be7450cd925e5e8", "verify_npo_objective.py": "769c11dcea34aa0f9355f7c9511f200cefcea8892f8bb5507358e476774ea853", "verify_npo_run.py": "fd334f8e53692e448d1ad643318bec38c9684873f950fcb9b3572f247304c594", "verify_tofu_official_eval.py": "d8645f8d99d14b6342f12e0fa677a5b47c6818fe2c1f431d1d4f583c17110979"}')
LAUNCH_CUTOFF_UTC = "2026-11-08T20:58:10+00:00"
INPUT_ARCHIVE_NAME = "tofu_8b_npo_lora_pilot_artifacts.zip"

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if OBJECTIVE_CHECK.get("status") != "PASS" or OBJECTIVE_CHECK.get("protocol_sha256") != EXPECTED_PROTOCOL_SHA256:
    raise RuntimeError("Embedded upstream-objective equivalence receipt is not valid for this protocol.")

archives = list(Path("/kaggle/input").rglob(INPUT_ARCHIVE_NAME))
if len(archives) != 1:
    raise RuntimeError(f"Attach exactly one Kaggle dataset containing {INPUT_ARCHIVE_NAME}; found {len(archives)}")
input_archive = archives[0]
archive_sha256 = sha256(input_archive)
extract_dir = Path("/kaggle/working/eval_recovery_input")
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)

with zipfile.ZipFile(input_archive) as archive:
    bad = archive.testzip()
    if bad is not None:
        raise RuntimeError(f"Input artifact ZIP CRC failed: {bad}")
    roots = set()
    for info in archive.infolist():
        member = PurePosixPath(info.filename)
        mode = info.external_attr >> 16
        if member.is_absolute() or ".." in member.parts or stat.S_ISLNK(mode):
            raise RuntimeError(f"Unsafe archive member rejected: {info.filename}")
        if member.parts:
            roots.add(member.parts[0])
    if len(roots) != 1:
        raise RuntimeError(f"Expected exactly one archived run root; found {sorted(roots)}")
    run_root_name = next(iter(roots))
    if not re.fullmatch(r"tofu_8b_npo_run_\d{8}T\d{6}Z", run_root_name):
        raise RuntimeError(f"Unexpected archived run-root name: {run_root_name}")
    archive.extractall(extract_dir)

RUN_ROOT = extract_dir / run_root_name
TRAIN_DIR = RUN_ROOT / "npo_train"
ADAPTER_DIR = TRAIN_DIR / "adapter"
EVAL_DIR = RUN_ROOT / "tofu_official_eval"
PROTOCOL_PATH = RUN_ROOT / "npo_lora_8b_pilot_protocol.json"
OBJECTIVE_PATH = Path("/kaggle/working/npo_objective_equivalence.json")
OBJECTIVE_PATH.write_text(json.dumps(OBJECTIVE_CHECK, indent=2) + "\n", encoding="utf-8")

if not RUN_ROOT.is_dir() or not TRAIN_DIR.is_dir() or not EVAL_DIR.is_dir():
    raise RuntimeError("The archive is missing its run, training, or baseline evaluation directory.")
if sha256(PROTOCOL_PATH) != EXPECTED_PROTOCOL_SHA256:
    raise RuntimeError("Archived protocol differs from the preregistered local protocol.")
for filename, expected_sha in EXPECTED_SOURCE_SHA256.items():
    source = RUN_ROOT / filename
    if not source.is_file() or sha256(source) != expected_sha:
        raise RuntimeError(f"Archived source differs from its verified source hash: {filename}")

receipt_path = TRAIN_DIR / "training_receipt.json"
if not receipt_path.is_file():
    raise RuntimeError("Training receipt is missing; refusing evaluation.")
training_receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
started_at = datetime.fromisoformat(training_receipt.get("started_at_utc", "").replace("Z", "+00:00"))
launch_cutoff = datetime.fromisoformat(LAUNCH_CUTOFF_UTC)
expected_model = PROTOCOL["base_checkpoint"]
expected_training = PROTOCOL["dataset"]["training"]
device_map = training_receipt.get("runtime", {}).get("device_map", {})
schedule = training_receipt.get("training_schedule", {})
if (training_receipt.get("status") != "TRAINING_COMPLETE"
        or training_receipt.get("global_steps") != 500
        or training_receipt.get("seed") != 42
        or training_receipt.get("adapter_saved") is not True
        or training_receipt.get("protocol_sha256") != EXPECTED_PROTOCOL_SHA256
        or started_at <= launch_cutoff
        or training_receipt.get("training_started") is not True
        or training_receipt.get("model", {}).get("repo_id") != expected_model["repo_id"]
        or training_receipt.get("model", {}).get("revision") != expected_model["revision"]
        or training_receipt.get("dataset", {}).get("repo") != PROTOCOL["dataset"]["repo_id"]
        or training_receipt.get("dataset", {}).get("revision") != PROTOCOL["dataset"]["revision"]
        or training_receipt.get("dataset", {}).get("forget10_sha256") != expected_training["forget_file"]["sha256"]
        or training_receipt.get("dataset", {}).get("retain90_sha256") != expected_training["retain_file"]["sha256"]
        or training_receipt.get("dataset", {}).get("forget10_rows") != 400
        or training_receipt.get("dataset", {}).get("retain90_rows") != 3600
        or training_receipt.get("training_script_sha256") != EXPECTED_SOURCE_SHA256["npo_lora_train.py"]
        or training_receipt.get("method", {}).get("beta") != PROTOCOL["objective"]["beta"]
        or training_receipt.get("method", {}).get("epochs") != PROTOCOL["adaptation"]["epochs"]
        or training_receipt.get("method", {}).get("lora_rank") != PROTOCOL["adaptation"]["lora_rank"]
        or training_receipt.get("tokenization", {}).get("truncated_examples") != 0
        or training_receipt.get("tokenization", {}).get("max_length_observed", 999999) > 512
        or training_receipt.get("runtime", {}).get("cpu_or_disk_offload") is not False
        or any(str(device).lower() in {"cpu", "disk"} for device in device_map.values())
        or training_receipt.get("gpu_names", []).count("Tesla T4") != 2
        or schedule.get("planned_optimizer_steps") != 500
        or schedule.get("planned_loss_trace_rows") != 4000):
    raise RuntimeError("Archived receipt does not prove the locked 500-step seed-42 pilot completed.")
trace_path = TRAIN_DIR / "loss_trace.jsonl"
trace_rows = [json.loads(line) for line in trace_path.read_text(encoding="utf-8").splitlines() if line.strip()]
if len(trace_rows) != 4000 or any(row.get("global_step", 0) not in range(1, 501) for row in trace_rows):
    raise RuntimeError(f"Loss trace does not contain the expected 4,000 records; found {len(trace_rows)}.")
actual_adapter_hashes = {p.name: sha256(p) for p in sorted(ADAPTER_DIR.iterdir()) if p.is_file()}
if not actual_adapter_hashes or actual_adapter_hashes != training_receipt.get("adapter_files_sha256"):
    raise RuntimeError("Archived adapter files do not match the training receipt hashes.")

baseline_manifest_path = RUN_ROOT / "baseline_inputs" / "npo_8b_baseline_manifest.json"
if not baseline_manifest_path.is_file() or sha256(baseline_manifest_path) != EXPECTED_BASELINE_MANIFEST_SHA256:
    raise RuntimeError("Archived 8B baseline manifest is missing or has the wrong hash.")
baseline_manifest = json.loads(baseline_manifest_path.read_text(encoding="utf-8"))
for relative, expected_sha in baseline_manifest["files"].items():
    source = RUN_ROOT / "baseline_inputs" / relative
    if not source.is_file() or sha256(source) != expected_sha:
        raise RuntimeError(f"Archived baseline input hash mismatch: {relative}")

for name in ("llama_3_1_8b_full_official_eval.json", "llama_3_1_8b_retain90_official_eval.json",
             "audit/tofu_readiness_audit.json"):
    if not (EVAL_DIR / name).is_file():
        raise RuntimeError(f"Archived evaluation directory is missing required baseline file: {name}")

existing_result = EVAL_DIR / "llama_3_1_8b_npo_lora_seed42_official_eval.json"
existing_comparison = EVAL_DIR / "npo_comparison.json"
needs_inference = not (existing_result.is_file() and existing_comparison.is_file())
gpu_names = []
if needs_inference:
    if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
        raise RuntimeError("The adapter evaluation is missing. Select Kaggle GPU T4 x2 to run it.")
    gpu_names = [torch.cuda.get_device_name(i) for i in range(2)]
    if gpu_names.count("Tesla T4") != 2:
        raise RuntimeError(f"This locked pilot requires two Tesla T4 GPUs; found {gpu_names}")

print({
    "preflight": "PASS",
    "input_archive_sha256": archive_sha256,
    "run_root": RUN_ROOT.name,
    "protocol_sha256": sha256(PROTOCOL_PATH),
    "training_status": training_receipt["status"],
    "optimizer_steps": training_receipt["global_steps"],
    "adapter_files": len(actual_adapter_hashes),
    "baseline_manifest_sha256": sha256(baseline_manifest_path),
    "gpus": gpu_names,
    "evaluation_already_complete": not needs_inference,
    "will_retrain": False,
}, flush=True)


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import zipfile
from datetime import datetime, timezone

def run_streamed(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return process.wait()

result_key = "llama_3_1_8b_npo_lora_seed42_official_eval.json"
comparison_path = EVAL_DIR / "npo_comparison.json"
adapter_result_path = EVAL_DIR / result_key
if adapter_result_path.is_file() and comparison_path.is_file():
    print("Complete evaluation artifacts already exist; skipping inference and proceeding to independent verification.", flush=True)
else:
    command = [sys.executable, str(RUN_ROOT / "evaluate_npo_adapter.py"),
               "--model", "8b", "--seed", "42", "--adapter-dir", str(ADAPTER_DIR),
               "--output-dir", str(EVAL_DIR)]
    print({"evaluation": "START", "cmd": command, "time_utc": datetime.now(timezone.utc).isoformat()}, flush=True)
    code = run_streamed(command, RUN_ROOT / "evaluation_recovery.stdout.log")
    if code:
        raise RuntimeError(f"Adapter evaluation failed with exit code {{code}}; log retained in output archive.")

verify_command = [sys.executable, str(RUN_ROOT / "verify_npo_run.py"), "--model", "8b",
                  "--training-dir", str(TRAIN_DIR), "--eval-dir", str(EVAL_DIR),
                  "--protocol", str(PROTOCOL_PATH), "--objective-check", str(OBJECTIVE_PATH)]
verify_code = run_streamed(verify_command, RUN_ROOT / "npo_verification_recovery.stdout.log")
if verify_code:
    raise RuntimeError(f"Independent run verification failed with exit code {{verify_code}}.")

verification_path = EVAL_DIR / "npo_verification.json"
verification = json.loads(verification_path.read_text(encoding="utf-8"))
if verification.get("overall_status") != "PASS":
    raise RuntimeError("Independent run verifier did not record overall_status=PASS.")
provenance = {
    "recovery_kind": "evaluation_only_no_retraining",
    "input_archive": str(input_archive),
    "input_archive_sha256": archive_sha256,
    "run_root": RUN_ROOT.name,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA256,
    "training_receipt_sha256": sha256(receipt_path),
    "adapter_files_sha256": actual_adapter_hashes,
    "verification_status": verification.get("overall_status"),
    "recovered_at_utc": datetime.now(timezone.utc).isoformat(),
}
(RUN_ROOT / "evaluation_recovery_provenance.json").write_text(json.dumps(provenance, indent=2) + "\n", encoding="utf-8")
output_archive = Path("/kaggle/working/tofu_8b_npo_eval_recovery_artifacts.zip")
if output_archive.exists():
    output_archive.unlink()
with zipfile.ZipFile(output_archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RUN_ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(RUN_ROOT.parent))
print({"recovery": "PASS", "overall_verification": verification["overall_status"],
       "primary_result": verification.get("primary_result"),
       "research_diagnostic_gate_status": verification.get("research_diagnostic_gate_status"),
       "utility_gates": verification.get("utility_gates"),
       "output_archive": str(output_archive), "output_bytes": output_archive.stat().st_size}, flush=True)
